# Publication data exercise

Three take-home tasks on publication data:

1. **Author productivity** — clean author/paper tables and plot how many authors have published *X* papers.
2. **Acknowledgement extraction** — use the Gemini API to pull names and reasons from an acknowledgement paragraph.
3. **Semantic Scholar sampling** — download 100 papers, cache the raw JSON, and plot the distribution of unique authors per paper.

## Setup

```bash
pip install -r requirements.txt
cp .env.example .env          # add your own keys (optional for task 3)
```

The notebook reads the CSV/Parquet files from this folder. Task 2 needs a
`GEMINI_API_KEY`; task 3 works without a key but is rate-limited. Keys are read
from environment variables only — never hard-code them.

> **Security note:** an earlier version of this notebook contained a committed
> Google API key. It has been removed here, but it still exists in the git
> history. Revoke that key in Google Cloud and see `docs/security.md`.


In [ ]:
# Task 1: author productivity
from collections import Counter
from pathlib import Path
import ast
import re

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path.cwd()
df_authors = pd.read_csv(DATA_DIR / "authors.csv")
df_papers = pd.read_parquet(DATA_DIR / "papers.parquet")

ORCID_LIKE = re.compile(r"^[0-9xX-]{15,25}$")


def clean_orcid(value):
    """Return a bare, canonical ORCID iD, or None for missing/invalid values."""
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    text = re.sub(r"^https?://[^/]+/", "", text).strip("/ ")
    text = re.sub(r"\s+", "", text)
    return text if ORCID_LIKE.match(text) else None


def as_author_list(value):
    """Authors may arrive as a list/array of dicts or as a serialized string."""
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    if isinstance(value, dict):
        return []
    try:
        return list(value)
    except TypeError:
        return []


df_authors["ORCID"] = df_authors["ORCID"].apply(clean_orcid)
df_authors = df_authors.dropna(subset=["ORCID"]).drop_duplicates(subset=["ORCID"])

pairs = []
for paper in df_papers.itertuples(index=False):
    for author in as_author_list(paper.Authors):
        if isinstance(author, dict):
            orcid = clean_orcid(author.get("ORCID"))
            if orcid:
                pairs.append((orcid, paper.PaperID))

df_author_papers = pd.DataFrame(pairs, columns=["ORCID", "PaperID"]).drop_duplicates()
papers_per_author = (
    df_author_papers.groupby("ORCID")["PaperID"]
    .nunique()
    .reindex(df_authors["ORCID"])
    .fillna(0)
    .astype(int)
)

counts = Counter(papers_per_author)
x, y = zip(*sorted(counts.items())) if counts else ([], [])
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel("Number of papers")
plt.ylabel("Number of authors")
plt.title("Distribution of papers per author")
plt.tight_layout()
plt.show()

print(f"Authors with a valid ORCID: {len(df_authors):,}")
print(f"Paper-author pairs: {len(df_author_papers):,}")


## Task 2:

Below is an ackowledgement statement often seen in scientific papers. Please use ChatGPT/Gemini API to extract who is acknowledged and for what. (You should be able to use Gemini API for free)

Output format: 

```python
[
    {
        'Name': XXX,
        'Reason': YYY,
    }, 
    ...
]
```


In [ ]:
acknowledgement = '\nI would like to express my deepest gratitude to Professor Sarah Johnson, whose insightful guidance and constant encouragement have been invaluable throughout this work.\nI am also grateful to Dr. Michael Chen, who generously shared his expertise in data analysis, and to Dr. Priya Natarajan, who provided constructive feedback that helped refine my ideas.\nSpecial thanks go to Emily Rodríguez for her support with coding assistance, which significantly contributed to the progress of this project.\nFinally, I wish to acknowledge the unwavering support of my colleagues and friends, especially Daniel Kim, whose discussions and encouragement kept me motivated.\n'

In [ ]:
# Task 2: extract acknowledged people with the Gemini API
import json
import os
import re

import requests

API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if not API_KEY:
    raise SystemExit(
        "Set GEMINI_API_KEY (for example in a local .env file) before running this cell."
    )

prompt = """Extract the names of people who are acknowledged and the reason for each.
Return only a JSON array in this shape:
[{"Name": "...", "Reason": "..."}]

Acknowledgement statement:
""" + acknowledgement

response = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent",
    headers={"Content-Type": "application/json", "X-goog-api-key": API_KEY},
    json={"contents": [{"parts": [{"text": prompt}]}]},
    timeout=60,
)
response.raise_for_status()
text = response.json()["candidates"][0]["content"]["parts"][0]["text"]
text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
print(json.dumps(json.loads(text), indent=2, ensure_ascii=False))


## Task 3

SemanticScholar is an database of research papers. Read the [API document](https://www.semanticscholar.org/product/api) and complete the following steps:

1. Scrape 100 papers
2. Save the raw data file so we can replicate the results
3. Clean the data using whatever reasonable means
4. Plot the distribution of number of unique authors per papers.

In [ ]:
# Task 3: sample 100 Semantic Scholar papers and plot unique authors per paper
import ast
import json
import os
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import requests

API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "").strip()
url = "https://api.semanticscholar.org/graph/v1/paper/search"
params = {
    "query": "artificial intelligence",
    "limit": 100,
    "fields": "paperId,authors",
    "offset": 0,
}
headers = {"x-api-key": API_KEY} if API_KEY else {}
response = requests.get(url, params=params, headers=headers, timeout=60)
if response.status_code != 200:
    raise RuntimeError(f"Semantic Scholar returned {response.status_code}: {response.text}")

papers = response.json().get("data", [])
with open("raw_papers.json", "w", encoding="utf-8") as handle:
    json.dump(papers, handle)

df_raw = pd.read_json("raw_papers.json")
df_raw.to_csv("raw_papers.csv", index=False)

df_papers = df_raw.dropna(subset=["paperId"]).drop_duplicates(subset=["paperId"]).copy()
df_papers["authors"] = df_papers["authors"].apply(
    lambda value: ast.literal_eval(value) if isinstance(value, str) else value
)
df_papers["num_unique_authors"] = df_papers["authors"].apply(
    lambda authors: len({a.get("authorId") for a in authors or [] if a.get("authorId")})
)
df_papers = df_papers[df_papers["num_unique_authors"] > 0]

distribution = Counter(df_papers["num_unique_authors"])
x, y = zip(*sorted(distribution.items())) if distribution else ([], [])
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel("Unique authors per paper")
plt.ylabel("Number of papers")
plt.title("Distribution of unique authors per paper")
plt.tight_layout()
plt.show()

print(f"Papers kept: {len(df_papers)}")
